Name: Jingxuan Wang

Student ID: 35330058

Kaggle Name: jingxuanwang123

In [2]:
#Part 1
#Question 1
getwd()
setwd("~/Desktop")
# Loading the dataset.
load_reg_train <- read.csv("regression_train.csv")

# Fit a linear regression model and the target variable is happiness.
model1 <- lm(happiness ~ ., data = load_reg_train)
model_summary <- summary(model1)


# Extract the coefficients from the summary.
# The coefficients table includes estimate, SE, t-value, p-value.
coef_table <- model_summary$coefficients

#Find predictors associated with happiness.
#Select predictors with p-values < 0.01
predictors <- coef_table[
    coef_table[,4] <0.01,
    ]

#Remove the intercept terms as the question ask us for predictors.
predictors <-predictors[
    rownames(predictors) != "(Intercept)",
             ]
# Remove the intercept from the coefficient table.
coef_new <- coef_table[
    rownames(coef_table) != "(Intercept)",
    ]

#Sort predictors by absolute t-values in descending order.
top <- coef_new[
    order(abs(coef_new[,3]), decreasing = TRUE),
    ]

# Select top 5 predictors.
top5 <- top[1:5, ]

# Automatically fetch and print the output.
cat("Predictors associated with happiness (p<0.01):\n")
print(rownames(predictors))

cat("\nTop5 predictors:\n")
print(top5)

[1] "/Users/wangjingxuan/Jingxuan Wang"

Predictors associated with happiness (p<0.01):
 [1] "income10k - 15k"                                     
 [2] "income120k - 150k"                                   
 [3] "income150k - 200k"                                   
 [4] "income15k - 20k"                                     
 [5] "income200k above"                                    
 [6] "income20k - 50k"                                     
 [7] "income50k - 80k"                                     
 [8] "income80k - 120k"                                    
 [9] "whatIsYourHeightExpressItAsANumberInMetresM180 - 185"
[10] "alwaysStressed"                                      

Top5 predictors:
                 Estimate Std. Error  t value     Pr(>|t|)
income80k - 120k 37.28064   1.660049 22.45755 1.419507e-74
income50k - 80k  28.70233   1.742010 16.47656 1.115904e-47
income200k above 20.03062   1.382642 14.48721 4.777672e-39
income20k - 50k  20.81995   1.587546 13.11455 2.606555e-33
income15k - 20k  15.40746   1.284859 11.

In [3]:
# Part1
# Question 2
# This function require an actual and predicted value, and then calculate the RMSE between them.
rmse <- function(actual, predicted){
    error <- actual - predicted
    rmse_value <- sqrt(mean(error^2))
    return (rmse_value)
    }

# Use the model model make in Q1 to predict the happiness
prediction <- predict(model1, newdata = load_reg_train)

#Extract the actual value from the dataset.
actual <- load_reg_train$happiness
# Calculate the RMSE.
result <- rmse(actual, prediction)
result

[1] 6.701688

In [4]:
# Part1
# Question 3
# Calculate the number of rows in dataset.
num <- nrow(load_reg_train)

# Perform the bidirectional stepwise regression using BIC.
# Variables can be both added and removed to find a simpler model with better BIC value.
step_model <- step(
    model1,
    direction= "both",
    k = log(num),
    trace = 0
    )

summary(step_model)

# Use the new model to predict.
new_predict <- predict(
    step_model,
    newdata = load_reg_train
    )
# Claculate the new rmse.
rmse_newp <- rmse(
    load_reg_train$happiness,
    new_predict
    )

rmse_newp

# Finding: After using the BIC method, a large number of predictors were removed but the rmse increased moderately from 6.70 to 7.33. 
# It indicates that maybe most of the variables in the original model are redundant and they may be less useful for predicting happiness.
# This suggests that a simpler model can also show the similar prediction performance and reduce the complexity of the model.


Call:
lm(formula = happiness ~ income + alwaysStressed + alwaysHaveFun + 
    alwaysSerious + alwaysDepressed + iFindMostThingsAmusing + 
    iUsuallyHaveAGoodInfluenceOnEvents, data = load_reg_train)

Residuals:
    Min      1Q  Median      3Q     Max 
-33.365  -4.587  -0.030   5.203  18.888 

Coefficients:
                                   Estimate Std. Error t value Pr(>|t|)    
(Intercept)                        -13.7027     0.6348 -21.587  < 2e-16 ***
income10k - 15k                      6.9401     1.2123   5.725 1.82e-08 ***
income120k - 150k                   12.4102     1.1859  10.465  < 2e-16 ***
income150k - 200k                   12.1301     1.1885  10.206  < 2e-16 ***
income15k - 20k                     14.0070     1.1820  11.851  < 2e-16 ***
income200k above                    20.6292     1.3292  15.519  < 2e-16 ***
income20k - 50k                     22.1535     1.4718  15.052  < 2e-16 ***
income50k - 80k                     29.0212     1.6938  17.133  < 2e-16 ***
incom

[1] 7.330305

In [5]:
#Part1
#Question4

# Get all predictors name, exclude happiness
predictorname <- names(load_reg_train)
predictorname <- predictorname[predictorname!="happiness"]

# Initial the best rmse.
# Create empty best_prediction and best_model to store the information.
rmse1 <- Inf
best_prediction <- NULL
best_model <- NULL

# Test all kind of combination with 2 predictors.
for (i in 1:(length(predictorname)-1)){

    for( j in (i+1): length(predictorname)){

        # Create a regression formula with 2 predictors.
        regression2 <- paste(
            "happiness ~",
            predictorname[i],
            "+",
            predictorname[j]
            )

        # Fit the model.
        model <- lm(as.formula(regression2),
                   data = load_reg_train)

        # Prediction
        prediction <- predict(model,
                              newdata = load_reg_train)

        # Calculate the rmse of this model.
        nrmse <- rmse(load_reg_train$happiness,
                      prediction)

        # Update the best model if the current rmse is less than the original rmse.
        if(nrmse<rmse1){
            rmse1 <- nrmse
            best_prediction <-c(predictorname[i], predictorname[j])
            best_model <- model
            }
        }
    }

# Fit the full model using all predictions.
all_prediction <- predict(
    model1,
    newdata = load_reg_train
    )

# Calculate the rmse of full model.
all_rmse <- rmse(
    load_reg_train$happiness,
    all_prediction)

     # Output
     best_prediction
     rmse1
     all_rmse


     
# Explanation: From the output, we can see the best 2 predictors are income and alwaysStressed, they achieved an rmse of 7.8854
#,compare with the full model rmse which is 6.7017.


[1] "income"         "alwaysStressed"

[1] 7.885411

[1] 6.701688

During the Part1 Question5, I used 5 different models including BIC-selected linear regression, Full linear regression, Random Forest, GBM Regression and XGBOOST. Based on their performance, I selected GBM Regression and XGBOOST as the submission in Kaggle. The comparision of the model is in the end of this Question.

In [6]:
# part1
# Question5
# 1. BIC-selected linear regression.
# Public score: 6.4521
fin.mod <- step_model
test <- read.csv("regression_test.csv")

# Predict happiness for the test data
pred.label <- predict(fin.mod, test)

# Export the file
write.csv(
    data.frame(
        "RowIndex" = seq(1, length(pred.label)),
        "Prediction" = pred.label
        ),
    "RegressionPredictLabel.csv",
    row.names = FALSE
    )


In [7]:
# part1
# Question5
# 2. Full linear regression
# Public score: 5.9530
fin.mod <- model1
test <- read.csv("regression_test.csv")

# Predict happiness for the test data
pred.label <- predict(fin.mod, test)

# Export the file
write.csv(
    data.frame(
        "RowIndex" = seq(1, length(pred.label)),
        "Prediction" = pred.label
        ),
    "RegressionPredictLabel.csv",
    row.names = FALSE
    )

In [8]:
# part1
# Question5
# 3.Random Forest
# Public score: 7.0828
# install.packages("randomForest")
library(randomForest)

set.seed(123)

load_reg_train <- read.csv("regression_train.csv")
test <- read.csv("regression_test.csv")

# Build the model
rf_model <- randomForest(
    happiness ~ .,
    data = load_reg_train,
    ntree = 500,
    importance = TRUE)

fin.mod <- rf_model

# Predict happiness for the test data
pred.label <- predict(fin.mod, test)

# Export the file
write.csv(
    data.frame(
        RowIndex = seq(1, length(pred.label)),
        Prediction = pred.label
        ),
    "RegressionPredictLabel.csv",
    row.names = F
    )



randomForest 4.7-1.2

Type rfNews() to see new features/changes/bug fixes.



In [10]:
# Part1 
# Question 5
# 4, GBM Regression
# Publc score: 5.27704
#getwd()
#setwd("~/Desktop")
library(gbm)

# Load the data
load_reg_train <- read.csv("regression_train.csv")
test <- read.csv("regression_test.csv")

# Cnvert all character variables to factors.
load_reg_train[] <- lapply(
    load_reg_train,
    function(x){
        if(is.character(x)) as.factor(x) else x
    }
)

test[] <- lapply(
    test,
    function(x){
        if(is.character(x)) as.factor(x) else x
    }
)

set.seed(123)

# Build the model
gbm_model <- gbm(
    happiness ~ .,
    data = load_reg_train,
    distribution = "gaussian",
    n.trees = 3000,
    interaction.depth = 5,
    shrinkage = 0.01,
    n.minobsinnode = 10,
    bag.fraction = 0.8,
    verbose = FALSE
)

# Do the prediction
pred.label <- predict(
    gbm_model,
    newdata = test,
    n.trees = 3000
)

# Export the file.
write.csv(
    data.frame(
        RowIndex = seq_len(nrow(test)),
        Prediction = pred.label
        ),
    "RegressionPredictLabel.csv",
    row.names = F
    )

In [11]:
# Part1 
# Question 5
# 5, XGBOOST
# Publc score: 4.65725

# install.packages("xgboost")
library(xgboost)

# Load the data
load_reg_train <- read.csv("regression_train.csv")
test <- read.csv("regression_test.csv")

# Cnvert all character variables to factors.
load_reg_train[] <- lapply(
    load_reg_train,
    function(x){
        if(is.character(x)) as.factor(x) else x
    }
)

test[] <- lapply(
    test,
    function(x){
        if(is.character(x)) as.factor(x) else x
    }
)

train_x <- model.matrix(
    happiness ~ .,
    data = load_reg_train
)[,-1]

train_y <- load_reg_train$happiness

test_x <- model.matrix(
    ~ .,
    data = test
)[,-1]

set.seed(123)

xgb_model <- xgboost(
    x = train_x,
    y = train_y,
    objective = "reg:squarederror",
    nrounds = 300,
    max_depth =4,
    learning_rate = 0.05,
    subsample = 0.8,
    colsample_bytree = 0.8)

# Do the prediction
pred.label <- predict(
    xgb_model,
    test_x
)

# Export the file.
write.csv(
    data.frame(
        RowIndex = seq_len(nrow(test)),
        Prediction = pred.label
        ),
    "RegressionPredictLabel.csv",
    row.names = F
    )



During Part1 Question5 I compared 5 different learning models.
First, I used a BIC selected model that variables can be both added and removed to find a simpler model with better BIC value. this model achieved the kaggle public score is 6.4521.
Second, I used full linear regression model, a linear regression that using all predictors to predict happiness.This model achieved a lower score which is 5.9530.
Third, I used the Random Forest to build the model and the performance is relatively poor, it achieved a public score of 7.0828.
Then I used GBM Regression to build the model because boosting method often provide better performance by combining mutiple week learners, this model achieved the publc score is 5.27704.
Finally, I used XGBOOST and achieved the highest publc score which is 4.65725.
Overall, the GBM Regression and XGBOOST were selected as the final submission on Kaggle.


The next is Part2, In part2, I used 3 different models including Random Forest, Naive Bayes and KNN. Based on their performance, I selected Naive Bayes and KNN as the submission in Kaggle. The comparision of the model is in the end of this Question.

In [12]:
# Part2 Classification 
# 1. Random Forest
# public score: 0.28290
# getwd()
# setwd("~/Desktop")
# Ramdom Forest
library(randomForest)

# Load the data
train_rf <- read.csv("classification_train.csv")
test_rf <- read.csv("classification_test.csv")

# Convert the predict variables to factor
train_rf$alwaysAnxious <- as.factor(train_rf$alwaysAnxious)

set.seed(123)
rf_model <- randomForest(
    alwaysAnxious ~ .,
    data = train_rf,
    ntree =  500)     
    


# Build the model
fin.mod <- rf_model

# Do the prediction.
pred.label <- predict(fin.mod, test_rf)
# put these predicted labels in a csv file that you can use to commit to the Kaggle Leaderboard
write.csv(
    data.frame("RowIndex" = seq(1, length(pred.label)), "Prediction" = pred.label),  
    "ClassificationPredictLabel.csv", 
    row.names = F
)

In [13]:
# Part 2 Classification 
# 2. Navie Bayes(Improved)
# public score: 0.38095

# Load package
library(e1071)

# Load the data
train_nb2 <- read.csv("classification_train.csv")
test_nb2 <- read.csv("classification_test.csv")

# Convert variables to factors
train_nb2[] <- lapply(train_nb2, as.factor)
test_nb2[] <- lapply(test_nb2, as.factor)

# Build the model
model_nb2 <- naiveBayes(
    alwaysAnxious ~ .,
    data = train_nb2,
    laplace = 1
    )

# Set the model
fin.model <- model_nb2

# Predict
pred.label <- predict(
    fin.model,
    test_nb2
    )

# Export csv file 
write.csv(
    data.frame(
        RowIndex = seq(1, length(pred.label)),
        Prediction = pred.label
        ),
    "ClassificationPredictLabel.csv", 
    row.names = F
)
    


In [14]:
# Part 2 Classification 
# 3. KNN
# public score :0.45261
# getwd()
# setwd("~/Desktop")
set.seed(123)
library(class)

# Load the data
train_kn <- read.csv("classification_train.csv")
test_kn <- read.csv("classification_test.csv")

# Convert variables to factor
train_kn$alwaysAnxious <- as.factor(train_kn$alwaysAnxious)

# Remove target variables
train_kn_x <- train_kn[, names(train_kn) != "alwaysAnxious"]

#-------------
test_kn_x <- test_kn

combined <- rbind(
    train_kn_x,
    test_kn_x
    )


combined[] <- lapply(
    combined,
    function(x){
        if(is.character(x) || is.factor(x)){
            as.numeric(as.factor(x))
            } else {
            x
            }
        }
    )


# Split back
train_kn_x <- combined[1:nrow(train_kn),]
test_kn_x <-  combined[(nrow(train_kn) + 1): nrow(combined),]


# Stanardization   
#----------

train_kn_scaled <- scale(train_kn_x)
test_kn_scaled <- scale(
    test_kn_x,
    center = attr(train_kn_scaled, "scaled:center"),
    scale = attr(train_kn_scaled, "scaled:scale"))


# KNN
pred.label <- knn(
    train = train_kn_scaled,
    test = test_kn_scaled,
    cl = train_kn$alwaysAnxious,
    k = 5
    )

# Check prediction distribution

table(pred.label)

# Export csv file 
write.csv(
    data.frame(
        RowIndex = seq_along(pred.label),
        Prediction = pred.label
        ),
    "ClassificationPredictLabel.csv", 
    row.names = F
)
    
    

pred.label
-2 -1  0  1  2 
 1 10 45 37  2 

The summary of Part2
During this question, I tried many classification methods. After comparing them, 
Then I keep three models of the method that can be reproducible, they are Random Forest, Navie Bayes and KNN 
Among them, KNN achieved the highest Kaggle public scores, which is 0.45261, then is Naive Bayes, which is 0.38095
So I selected this 2 results as the final submission in Kaggle.